# Evaluation sample for hand-labelling

## Load suppliers

In [1]:
import duckdb
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.datavalidation import DataValidation

CATEGORIES = [
    "Social Care", "Education & Childcare", "Construction & Property", "IT & Telecoms",
    "Energy & Utilities", "Transport & Travel", "Waste & Environment", "Professional Services",
    "Staffing & Recruitment", "Housing & Accommodation", "Public Sector Bodies",
    "Community & Voluntary", "Culture, Leisure & Events", "Goods & Supplies", "Other",
]
SAMPLE_SIZE = 150
SEED = 42

con = duckdb.connect("../council.duckdb")
sup = con.sql("""
    SELECT r.vendor_raw, r.main_expense_type, r.n_payments, r.total_spend,
           r.std_name_rules, l.std_name_llm_v1
    FROM supplier_rules r
    JOIN supplier_llm_v1 l USING (vendor_raw)
""").df()
con.close()
len(sup)

2529

## Sample 150 suppliers
20 spelling-variant groups (to test name standardisation) plus random suppliers.

In [2]:
llm_groups = sup.groupby("std_name_llm_v1")["vendor_raw"].transform("count")
rule_groups = sup.groupby("std_name_rules")["vendor_raw"].transform("count")
sup["in_variant_group"] = (llm_groups > 1) | (rule_groups > 1)

variant_keys = (sup[sup["in_variant_group"]]["std_name_llm_v1"]
                .drop_duplicates()
                .sample(n=20, random_state=SEED))
variant_rows = (sup[sup["std_name_llm_v1"].isin(variant_keys)]
                .groupby("std_name_llm_v1").head(3))

rest = sup[~sup["vendor_raw"].isin(variant_rows["vendor_raw"])]
random_rows = rest.sample(n=SAMPLE_SIZE - len(variant_rows), random_state=SEED)

sample = (pd.concat([variant_rows, random_rows])
          .sort_values("vendor_raw", key=lambda s: s.str.upper())
          .reset_index(drop=True))
sample.insert(0, "id", range(1, len(sample) + 1))

print("from spelling-variant groups:", len(variant_rows))
print("random:", len(random_rows))
print("total:", len(sample))

from spelling-variant groups: 41
random: 109
total: 150


## Write the labelling sheet

In [3]:
wb = Workbook()
ws = wb.active
ws.title = "labels"

headers = ["id", "vendor_raw", "expense_type", "n_payments", "total_spend",
           "true_standard_name", "true_category", "is_person", "notes"]
ws.append(headers)
for r in sample.itertuples():
    ws.append([r.id, r.vendor_raw, r.main_expense_type, r.n_payments, float(r.total_spend),
               None, None, None, None])

fill = PatternFill("solid", fgColor="FFF2CC")
for col in "FGHI":
    for row in range(2, len(sample) + 2):
        ws[f"{col}{row}"].fill = fill
for cell in ws[1]:
    cell.font = Font(bold=True)
for col, width in zip("ABCDEFGHI", [5, 38, 32, 11, 13, 38, 26, 10, 30]):
    ws.column_dimensions[col].width = width
for row in range(2, len(sample) + 2):
    ws[f"E{row}"].number_format = "£#,##0"
ws.freeze_panes = "C2"

lists = wb.create_sheet("lists")
for i, c in enumerate(CATEGORIES, start=1):
    lists[f"A{i}"] = c
lists["C1"], lists["C2"] = "Y", "N"

dv_cat = DataValidation(type="list", formula1=f"=lists!$A$1:$A${len(CATEGORIES)}", allow_blank=True)
dv_person = DataValidation(type="list", formula1="=lists!$C$1:$C$2", allow_blank=True)
ws.add_data_validation(dv_cat)
ws.add_data_validation(dv_person)
dv_cat.add(f"G2:G{len(sample) + 1}")
dv_person.add(f"H2:H{len(sample) + 1}")

guide = wb.create_sheet("how_to_label", 0)
rules = [
    "How to label (fill the yellow columns on the 'labels' sheet)",
    "",
    "true_standard_name: the organisation's real name as you would write it, e.g. 'Liverpool John Moores University'.",
    "   - Same organisation = exactly the same name, even if the raw spellings differ.",
    "   - Finish names that were cut off at 35 characters if you can (Google it if unsure).",
    "   - Drop Ltd / Limited / PLC.",
    "true_category: pick from the dropdown. Choose by what the organisation mainly does.",
    "is_person: Y if it is a private individual, otherwise N (or leave blank).",
    "notes: optional, e.g. 'unsure' or 'could also be Social Care'.",
    "",
    "Label from the name and expense type only. Don't look at the rule or LLM answers first,",
    "otherwise the accuracy numbers will be biased.",
    "",
    "Save the file with the same name when you're done (File > Save, keep .xlsx format).",
]
for i, line in enumerate(rules, start=1):
    guide[f"A{i}"] = line
guide["A1"].font = Font(bold=True, size=13)
guide.column_dimensions["A"].width = 110

import os
os.makedirs("../data/labels", exist_ok=True)
OUT = "../data/labels/labelling_sheet.xlsx"
if os.path.exists(OUT):
    raise FileExistsError(f"{OUT} already exists - delete it first if you really want a new sample")
wb.save(OUT)
print("saved", OUT)

saved ../data/labels/labelling_sheet.xlsx
